# StringForge Quickstart

This notebook shows the shortest StringForge-first path through the public release: query a Calabi-Yau catalogue, load a cached row into the JAXVacua bridge, and identify where persistent vacua are stored.  StringForge owns the database, cache/offline, model-loading bridge, and vault conventions.  JAXVacua owns the flux-vacuum physics once a model has been constructed.

The examples below are intentionally small.  They are meant to establish the boundary between infrastructure and physics, not to run a full vacuum search.


## 1. Create a database handle

`LCSDatabase(dataset="tdf")` presents the toric database in the mirror convention used by `jaxvacua.lcs.lcs_tree`.  Construction itself is lightweight; the catalogue and shards are fetched lazily when they are first requested and then cached locally.


In [ ]:
from stringforge import LCSDatabase

# Uses ./.stringforge_cache by default unless STRINGFORGE_DATA_DIR is set.
db = LCSDatabase(dataset="tdf")


## 2. Query a tiny catalogue slice

The query returns a `pandas.DataFrame`.  At this stage no JAXVacua object has been built; this is pure catalogue infrastructure and is the right level for browsing, filtering, batching, or preparing HPC jobs.


In [ ]:
catalogue = db.query(has_conifolds=True).head(3)
columns = [
    c for c in ("h11", "h12", "ks_id", "triang_id", "n_conifolds", "has_gv")
    if c in catalogue.columns
]
catalogue[columns]


## 3. Load one row as `lcs_tree`

`db.load(...)` turns one database row into the topological data container used by JAXVacua.  Passing `h11` and `h12` from the catalogue row makes the lookup unambiguous when the same `ks_id` occurs in several Hodge-number sectors.


In [ ]:
if catalogue.empty:
    raise RuntimeError("The demonstration query returned no rows; choose a different filter.")

row = catalogue.iloc[0]
tree = db.load(
    ks_id=int(row["ks_id"]),
    triang_id=int(row["triang_id"]),
    h11=int(row["h11"]),
    h12=int(row["h12"]),
    include_gv=False,
    include_conifolds=False,
)

type(tree).__name__, tree.h11, tree.h12


## 4. Load the same row as a JAXVacua finder

`db.load_model(...)` is the bridge into JAXVacua.  StringForge still supplies the data and conventions; the returned object is a JAXVacua `FluxVacuaFinder`, so vacuum search, ISD sampling, flux bounding, Hessians, and mass spectra should be followed in the JAXVacua documentation.


In [ ]:
finder = db.load_model(
    ks_id=int(row["ks_id"]),
    triang_id=int(row["triang_id"]),
    h11=int(row["h11"]),
    h12=int(row["h12"]),
    include_gv=False,
    include_conifolds=False,
)

type(finder).__name__


## 5. Persistent records: VacuaWriter (curated) and Vulcan (production)

StringForge ships two writers, chosen by volume and curation level:

- **`VacuaWriter`** -- low-volume, curated vacua headed into `vacua_vault`.
  One HuggingFace commit per `designate_vacua` call. Use this for the
  entries that back a paper.
- **`Vulcan`** -- high-volume cluster output. Workers stage parquet
  shards locally; a separate sync tier batches them into HuggingFace
  commits (default up to 500 files per commit, under a 90-commit/hour
  rolling-window budget). Use this for cluster runs you will later
  filter and ML-train against. See the
  [Vulcan cluster-runs tutorial](database_and_infrastructure/vulcan_cluster_runs)
  for the worked example.

The cell below shows the minimal VacuaWriter pattern and a one-liner
Vulcan handle. Both are commented out so the quickstart does not push
to HuggingFace.


In [ ]:
from stringforge import set_vault_dir

set_vault_dir("./vacua_vault")

# Typical downstream pattern, after JAXVacua has produced a DataFrame of solutions:
# db.designate_vacua(vacua_df, model=finder, label="benchmark_scan", committed_by="A. Schachner")
# db.push_vacua_to_hub(create_pr=True)

# Vulcan: production-tier writer for cluster output.
# from stringforge.vulcan import Vulcan
# import tempfile
# forge = Vulcan(repo="user/quickstart_demo", staging_dir=tempfile.mkdtemp())
# forge.list_pending()   # empty until forge.write(...) is called


## Where to go next

- For catalogue browsing, offline caches, batch loading, and shard behaviour, continue with the database tutorial.
- For persistent vacua, validation, designation, and HuggingFace upload/fetch workflows, continue with the vault tutorials.
- For production-scale cluster runs, batched HuggingFace uploads, and
  the read/ML-view side of forged shards, continue with the
  [Vulcan cluster-runs tutorial](database_and_infrastructure/vulcan_cluster_runs).
- For full flux-vacuum physics, use the JAXVacua tutorials after constructing a `FluxVacuaFinder` with `db.load_model(...)`.
- For specialised KKLT-style curated subsets, read the advanced KKLT database page after the basic TDF/CICY workflow is clear.
